# LLM 예제 및 실습

이 노트북은 **`질의 응답`, `도메인 특화 챗봇` 기초 예제 실습**을 수행하는 노트북입니다.

✅ 내용  
1. Llama를 활용한 질의 응답 예제
2. Llama와 RAG를 활용한 질의 응답 예제
3. Llama 기반의 의료 분야 특화 챗봇 예제

In [8]:
%conda env create -f llm.yaml -n llm

Retrieving notices: done

Note: you may need to restart the kernel to use updated packages.



CondaValueError: prefix already exists: C:\Users\ceo\miniforge3\envs\llm



### Llama 3.2를 활용한 질의 응답 예제
사전학습된 Llama 3.2 Korean Bllossom 모델과 토크나이저를 로드하여 한국어 질문에 대한 자연어 답변을 생성합니다. 

GPU 사용 가능 여부를 자동 감지하고 텍스트 생성 파라미터를 조정하여 질의응답 형식으로 응답을 생성하고 후처리합니다.

* CUDA GPU 사용 가능 여부를 확인하여 자동으로 디바이스를 설정하고 transformers 라이브러리에서 Bllossom/llama-3.2-Korean-Bllossom-3B 모델과 토크나이저 로드
* 질문을 "질문: {question}\n답변:" 형식의 프롬프트로 변환하고 토크나이저로 인코딩하여 CUDA 디바이스에 입력 준비
* top_p, top_k, temperature 등의 생성 파라미터를 설정하여 모델로 텍스트 생성 수행하고 반복 방지 옵션 적용
* 생성된 응답을 디코딩하고 원본 질문 부분을 제거하여 순수 답변만 추출 및 출력

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 기본 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Llama 모델 및 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(
    'Bllossom/llama-3.2-Korean-Bllossom-3B'
)

model = AutoModelForCausalLM.from_pretrained(
    'Bllossom/llama-3.2-Korean-Bllossom-3B', 
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
    offload_folder="./offload"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def answer_question(question, model, tokenizer, max_length=128):
    # 질의응답용 프롬프트 형식 지정
    prompt = f"질문: {question}\n답변:"
    
    # 답변 생성
    model_device = next(model.parameters()).device
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(model_device)
    
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=max_length,
            num_return_sequences=1,
            do_sample=True,
            top_p=0.92,
            top_k=50,
            temperature=0.7,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # 생성된 답변 디코딩
    answer = tokenizer.decode(output[0], skip_special_tokens=True)
    
    # 답변 부분만 추출 (원래 질문 제거)
    answer = answer.split("답변:")[1].strip()
    
    return answer

# 사용 예제
question = "삼성전자는 어떤 회사인가요?"
answer = answer_question(question, model, tokenizer)
print(f"질문: {question}")
print(f"답변: {answer}")

loading file tokenizer.json from cache at C:\Users\ceo\.cache\huggingface\hub\models--Bllossom--llama-3.2-Korean-Bllossom-3B\snapshots\e68fbb0d9c2a4031b0d61b14014eac1a4810ac2e\tokenizer.json
loading file tokenizer.model from cache at None
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at C:\Users\ceo\.cache\huggingface\hub\models--Bllossom--llama-3.2-Korean-Bllossom-3B\snapshots\e68fbb0d9c2a4031b0d61b14014eac1a4810ac2e\special_tokens_map.json
loading file tokenizer_config.json from cache at C:\Users\ceo\.cache\huggingface\hub\models--Bllossom--llama-3.2-Korean-Bllossom-3B\snapshots\e68fbb0d9c2a4031b0d61b14014eac1a4810ac2e\tokenizer_config.json
loading file chat_template.jinja from cache at None
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
loading configuration file config.json from cache at C:\Users\ceo\.cache\huggingface\hub\models--Bllossom--llama-3.2-Korean-Bll

질문: 삼성전자는 어떤 회사인가요?
답변: 삼성전자는 전자제품을 제조하고 판매하는 회사입니다. 주요 제품으로는 TV, 모바일 기기, 전자기기 등이 있습니다. 주요 업체로는 LG, SKH, 소니 등이 있으며, 전 세계적으로 다양한 시장에서 활동하고 있습니다. 삼성전자에는 삼성전자, 삼성SDI, 삼성DS, 삼성디스플레이 등 다양한 자회사들이 있으며, 다양한 제품과 서비스를 제공합니다. 삼성전자의 주요 제품은 다음과 같습니다:
- TV: QLED, SUHD, UHD


### Llama와 RAG를 활용한 질의 응답 예제
한국어 문서 데이터를 임베딩하여 FAISS 벡터 인덱스를 구성하고 질문과 유사한 문서를 검색합니다. 

검색된 문서를 컨텍스트로 활용하여 한국어 생성 모델이 정확한 답변을 생성하는 RAG(Retrieval-Augmented Generation) 파이프라인을 구현합니다.

* ko-sbert-multitask 임베딩 모델을 로드하고 한국어 문서 10개를 벡터화하여 FAISS IndexFlatL2 인덱스에 저장
* Bllossom llama-3.2-Korean-Bllossom-3B 생성 모델을 로드하고 채팅 템플릿 형식으로 프롬프트 구성
* 사용자 질문을 임베딩하여 FAISS 인덱스에서 상위 k개 유사 문서를 L2 거리 기반으로 검색
* 검색된 문서들을 컨텍스트로 제공하고 temperature, top_p 등의 파라미터를 조정하여 한국어 답변 생성
* 대한민국, K-POP, 전통 스포츠 등에 관한 테스트 질문으로 RAG 시스템의 검색 및 생성 성능 평가

In [2]:
import torch
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

# 기본 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

# 모델 설정
EMBEDDING_MODEL = "jhgan/ko-sbert-multitask"
GENERATOR_MODEL = "Bllossom/llama-3.2-Korean-Bllossom-3B"  # 한국어 특화 3B 모델

# 샘플 데이터
documents = [
    '대한민국은 동아시아 한반도 남부에 위치한 민주공화국이다.',
    '대한민국의 수도는 서울특별시이며, 인구 약 970만 명이다.',
    'K-POP은 한국 대중음악으로, BTS, 블랙핑크 등이 전 세계적으로 인기를 얻고 있다.',
    'K-드라마는 한국 드라마로, 오징어 게임, 킹덤 등이 글로벌 히트를 기록했다.',
    '한국의 전통 음식으로는 김치, 비빔밥, 불고기 등이 있다.',
    '한국의 전통 스포츠로는 태권도와 씨름이 있다.',
    '한국의 주요 산업으로는 반도체, 자동차, 조선 산업이 발달하였다.',
    '삼성전자, LG전자, 현대자동차 등이 대표적인 글로벌 기업이다.',
    '정보통신기술(ICT) 강국으로 5G 네트워크 인프라가 세계 최고 수준이다.',
    '제주도는 한국 최대의 섬으로 한라산과 자연경관으로 유명하다.'
]

dataset = Dataset.from_dict({"text": documents})
print(f"문서 수: {len(documents)}")

# 임베딩 모델 로드
print("임베딩 모델 로드 중...")
embed_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
embed_model = AutoModel.from_pretrained(EMBEDDING_MODEL).to(device)

def get_embeddings(texts):
    inputs = embed_tokenizer(texts, padding=True, truncation=True, 
                            return_tensors="pt", max_length=512).to(device)
    with torch.no_grad():
        outputs = embed_model(**inputs)
    
    # Mean pooling
    embeddings = outputs.last_hidden_state
    mask = inputs['attention_mask'].unsqueeze(-1).expand(embeddings.size()).float()
    pooled = torch.sum(embeddings * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
    return pooled.cpu().numpy()

# 문서 임베딩 생성
print("문서 임베딩 생성 중...")
doc_embeddings = get_embeddings(documents)

# FAISS 인덱스 생성
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings.astype(np.float32))
print(f"FAISS 인덱스 생성 완료: {index.ntotal}개 문서")

# 검색 함수
def search_documents(query, k=3):
    print(f"\n질문: {query}")
    query_embedding = get_embeddings([query])
    distances, indices = index.search(query_embedding.astype(np.float32), k)
    
    results = []
    print("검색된 문서:")
    for i, idx in enumerate(indices[0]):
        doc = documents[idx]
        results.append(doc)
        print(f"  {i+1}. {doc}")
    
    return results

# 생성 모델 로드
print("생성 모델 로드 중...")
gen_tokenizer = None
gen_model = None

try:
    gen_tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL)
    gen_model = AutoModelForCausalLM.from_pretrained(
        GENERATOR_MODEL,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        low_cpu_mem_usage=True,
        offload_folder="./offload"
    )
    
    if gen_tokenizer.pad_token is None:
        gen_tokenizer.pad_token = gen_tokenizer.eos_token
    
    print("생성 모델 로드 완료")
    
except Exception as e:
    print(f"모델 로드 실패: {e}")
    print("Hugging Face 토큰 설정이 필요하거나 모델 접근 권한이 없을 수 있습니다.")
    print("임베딩과 검색 기능은 사용 가능합니다.")

# 답변 생성 함수
def generate_answer(query, docs):
    if gen_model is None or gen_tokenizer is None:
        return "생성 모델이 로드되지 않아 답변을 생성할 수 없습니다. 검색된 문서를 참고하세요."
    
    context = "\n".join(docs)
    
    messages = [
        {"role": "system", "content": "주어진 문맥 정보를 바탕으로 한국어로 정확하고 간결하게 답변하세요. 문맥에 답이 없으면 '제공된 정보로는 답변할 수 없습니다'라고 답하세요."},
        {"role": "user", "content": f"문맥: {context}\n\n질문: {query}"}
    ]
    
    try:
        prompt = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        model_device = next(gen_model.parameters()).device
        inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model_device)
        
        with torch.no_grad():
            outputs = gen_model.generate(
                **inputs,
                max_new_tokens=128,  # 토큰 수 줄임
                temperature=0.3,     # 더 결정적으로
                top_p=0.8,          # 샘플링 범위 축소
                repetition_penalty=1.2,  # 반복 방지
                do_sample=True,
                pad_token_id=gen_tokenizer.eos_token_id,
                eos_token_id=gen_tokenizer.eos_token_id
            )
        
        response = gen_tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        # 반복된 내용이나 불필요한 부분 제거
        response = response.strip()
        
        # assistant나 기타 반복 패턴 제거
        if 'assistant' in response:
            response = response.split('assistant')[0].strip()
        
        # 첫 번째 완전한 문장들만 추출 (점이나 물음표로 끝나는)
        sentences = []
        for sent in response.split('.'):
            sent = sent.strip()
            if sent and not any(word in sent.lower() for word in ['assistant', 'user', 'system']):
                sentences.append(sent)
            if len(sentences) >= 3:  # 최대 3문장
                break
        
        if sentences:
            result = '. '.join(sentences)
            if not result.endswith('.'):
                result += '.'
            return result
        else:
            return response.split('\n')[0].strip()  # 첫 번째 줄만
    
    except Exception as e:
        return f"답변 생성 중 오류가 발생했습니다: {e}"

# RAG 파이프라인 실행
def ask_question(query):
    retrieved_docs = search_documents(query)
    answer = generate_answer(query, retrieved_docs)
    print(f"\n답변: {answer}")
    return answer

# 테스트 실행 함수
def run_test():
    print("\nRAG 시스템 테스트")
    print("-" * 40)
    
    test_questions = [
        "대한민국의 수도는 어디인가요?",
        "한국의 K-POP에 대해 알려주세요.",
        "한국의 전통 스포츠는 무엇인가요?",
        "제주도는 어디인가요?"
    ]
    
    for question in test_questions:
        ask_question(question)
        print("-" * 40)
    
    print("테스트 완료")

# 사용법 안내
print("\n사용법:")
print("ask_question('질문 내용')  # 개별 질문")
print("run_test()                # 전체 테스트 실행")

# 예시 실행 (주석 해제하여 사용)
run_test()

사용 장치: cuda
문서 수: 10
임베딩 모델 로드 중...
문서 임베딩 생성 중...
FAISS 인덱스 생성 완료: 10개 문서
생성 모델 로드 중...


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.66s/it]


생성 모델 로드 완료

사용법:
ask_question('질문 내용')  # 개별 질문
run_test()                # 전체 테스트 실행

RAG 시스템 테스트
----------------------------------------

질문: 대한민국의 수도는 어디인가요?
검색된 문서:
  1. 대한민국의 수도는 서울특별시이며, 인구 약 970만 명이다.
  2. 대한민국은 동아시아 한반도 남부에 위치한 민주공화국이다.
  3. K-POP은 한국 대중음악으로, BTS, 블랙핑크 등이 전 세계적으로 인기를 얻고 있다.

답변: 서울입니다.
----------------------------------------

질문: 한국의 K-POP에 대해 알려주세요.
검색된 문서:
  1. K-POP은 한국 대중음악으로, BTS, 블랙핑크 등이 전 세계적으로 인기를 얻고 있다.
  2. K-드라마는 한국 드라마로, 오징어 게임, 킹덤 등이 글로벌 히트를 기록했다.
  3. 한국의 전통 음식으로는 김치, 비빔밥, 불고기 등이 있다.

답변: 한국의 K-POP(Korean Pop)은 한국에서 태어난 음악을 의미하며, 주로 청소년과 젊은 세대에게 큰 인기를 끌고 있습니다. 대표적인 그룹으로는 BTS(Bangtan Sonyeondan), 블랙핑크(Blackpink) 등이 있으며, 이들은 전 세계적으로 많은 팬들을 모으고 있습니다. 또한, K-POP은 다양한 장르와 스타일을 가지고 있어, 현대적이고 동시대의 감정을 표현하는 데 강점을 가집니다.
----------------------------------------

질문: 한국의 전통 스포츠는 무엇인가요?
검색된 문서:
  1. 한국의 전통 스포츠로는 태권도와 씨름이 있다.
  2. 한국의 전통 음식으로는 김치, 비빔밥, 불고기 등이 있다.
  3. K-POP은 한국 대중음악으로, BTS, 블랙핑크 등이 전 세계적으로 인기를 얻고 있다.

답변: 태권도와 씨림입니다.
------------------------

### Llama 기반의 의료 분야 특화 챗봇 예제
llama2-ko-medical-7b 모델과 한국어 의료 대화 데이터셋을 로드하여 환자 증상에 대한 의료 상담 응답을 생성합니다. 

부적절한 표현을 필터링하고 응답을 정리하여 교육 및 연구 목적의 의료 챗봇 시스템을 구현합니다.

* squarelike/llama2-ko-medical-7b 모델과 토크나이저를 로드하고 ko_medical_chat 데이터셋을 불러와 의료 대화 샘플 확인
* 환자 증상을 입력받아 "환자의 증상: {symptoms}\n\n의사의 진단 및 조언:" 형식의 프롬프트로 구성하여 모델에 전달
* temperature, top_p, repetition_penalty 파라미터를 조정하여 의료 상담 응답을 생성하고 특수 토큰 제거
* 부적절한 표현 필터링, 연속 공백 제거, 완전한 문장 추출 등의 후처리를 통해 응답 품질 개선
* 데이터셋의 랜덤 샘플과 사용자 정의 증상으로 응답을 테스트하고 결과 출력

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import random

# 기본 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

# 모델 설정
model_name = "squarelike/llama2-ko-medical-7b"
print("모델 로드 중...")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        low_cpu_mem_usage=True,
        offload_folder="./offload"
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    print("모델 로드 완료")
    
except Exception as e:
    print(f"모델 로드 실패: {e}")

# 의료 데이터셋 로드
print("의료 데이터셋 로드 중...")
try:
    dataset = load_dataset("squarelike/ko_medical_chat", split="train")
    print(f"데이터셋 로드 완료: {len(dataset)}개 샘플")
    
except Exception as e:
    print(f"데이터셋 로드 실패: {e}")
    # 대체 데이터
    dataset = None

# 의료 상담 응답 생성
def generate_medical_response(symptoms, max_tokens=200):
    # 더 직접적인 의료 상담 프롬프트
    prompt = f"""환자의 증상: {symptoms}

의사의 진단 및 조언:"""
    
    try:
        model_device = next(model.parameters()).device
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model_device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=0.4,
                top_p=0.8,
                repetition_penalty=1.15,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
                # early_stopping 제거 (경고 메시지 방지)
            )
        
        # 응답 추출
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        # 응답 정리
        response = clean_response(response)
        
        return response if response.strip() else "적절한 응답을 생성할 수 없습니다."
        
    except Exception as e:
        return f"응답 생성 중 오류: {e}"

def clean_response(text):
    """응답 텍스트 정리 함수"""
    import re
    
    # 1. 기본 정리
    text = text.strip()
    
    # 2. 불필요한 특수문자 및 태그 제거
    remove_patterns = [
        '</s>', '<s>', '<<', '>>', '[INST]', '[/INST]', 
        'SYS', '환자:', '의사:', '▶', '①', '②', '③'
    ]
    
    for pattern in remove_patterns:
        text = text.replace(pattern, ' ')
    
    # 3. 연속된 공백을 하나로 통합
    text = re.sub(r'\s+', ' ', text)
    
    # 4. 부적절한 표현 필터링
    inappropriate_words = ['뚱뚱', '못생', '이상하', '바보', '멍청']
    for word in inappropriate_words:
        if word in text:
            return "증상에 대한 정확한 진단을 위해 병원 진료를 받으시기 바랍니다."
    
    # 5. 질문으로 시작하는 경우 제거
    if text.startswith(('왜', '어떻게', '언제', '무엇', '어디서')):
        sentences = text.split('.')
        if len(sentences) > 1:
            text = '.'.join(sentences[1:]).strip()
    
    # 6. 완전한 문장 추출 (최대 2문장)
    sentences = []
    for sentence in text.split('.'):
        sentence = sentence.strip()
        if len(sentence) > 10 and '?' not in sentence:
            sentences.append(sentence)
            if len(sentences) >= 2:
                break
    
    if sentences:
        result = '. '.join(sentences) + '.'
    else:
        # 문장이 없으면 원본의 처음 부분만
        words = text.split()[:20]
        result = ' '.join(words)
        if result and not result.endswith(('.', '!', '?')):
            result += '.'
    
    # 7. 최종 정리
    result = result.replace('..', '.').strip()
    
    # 8. 너무 짧은 경우 기본 메시지
    if len(result) < 15:
        return "증상에 대한 정확한 진단을 위해 병원 진료를 받으시기 바랍니다."
    
    return result

# 데이터셋 샘플 테스트
def test_dataset_samples():
    print("\n데이터셋 샘플 테스트")
    print("-" * 40)
    
    if dataset is None:
        print("데이터셋을 사용할 수 없습니다.")
        return
    
    # 랜덤 샘플 3개 테스트
    indices = random.sample(range(len(dataset)), min(3, len(dataset)))
    
    for i, idx in enumerate(indices, 1):
        sample = dataset[idx]
        conversations = sample.get('conversations', [])
        
        if conversations:
            # 환자의 첫 번째 메시지
            client_messages = [conv['value'] for conv in conversations if conv['from'] == 'client']
            if client_messages:
                patient_symptom = client_messages[0]
                
                print(f"\n테스트 {i}")
                print(f"환자 증상: {patient_symptom}")
                
                ai_response = generate_medical_response(patient_symptom)
                print(f"AI 응답: {ai_response}")

# 사용자 정의 테스트
def test_custom_symptoms():
    print("\n사용자 정의 증상 테스트")
    print("-" * 40)
    
    symptoms = [
        "머리가 아프고 열이 나요",
        "기침이 2주째 계속되고 있어요", 
        "가슴이 답답하고 숨이 차요"
    ]
    
    for i, symptom in enumerate(symptoms, 1):
        print(f"\n테스트 {i}")
        print(f"증상: {symptom}")
        response = generate_medical_response(symptom)
        print(f"AI 응답: {response}")

# 실행
if __name__ == "__main__":
    print("한국어 의료 챗봇 예제")
    print("=" * 40)
    print("주의: 교육/연구 목적이며 실제 의료 진단을 대체할 수 없습니다.")
    print("=" * 40)
    
    test_dataset_samples()
    test_custom_symptoms()
    
    print("\n개별 테스트:")
    print("generate_medical_response('증상') 함수 사용")

# 사용 예시
# response = generate_medical_response("목이 아프고 기침이 나요")
# print(response)

c:\Users\ceo\miniforge3\envs\llm10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


사용 장치: cuda
모델 로드 중...


Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.60s/it]


모델 로드 완료
의료 데이터셋 로드 중...
데이터셋 로드 완료: 3038개 샘플
한국어 의료 챗봇 예제
주의: 교육/연구 목적이며 실제 의료 진단을 대체할 수 없습니다.

데이터셋 샘플 테스트
----------------------------------------

테스트 1
환자 증상: 안녕하세요. 저는 G.C.라고 합니다. 65세이고, 은퇴한 정신과 간호사입니다. 30년 동안 당뇨병이 있지만 인슐린을 맞지 않았습니다. 작년 11월에 오른쪽 무릎을 교체 수술했습니다. 요가를 얼마나 할 수 있을까요? 아유르베다나 침술이 도움이 될까요?
cuda:0
AI 응답: 정형외과 전문의 이우석입니다. 요가는 유연성을 기르는 데 도움이 됩니다.

테스트 2
환자 증상: 저는 피로와 숨이 차다는 증상이 있어요.
cuda:0
AI 응답: 폐결핵은 폐에 생긴 병으로, 결핵균(mycobacterium tuberculosis)이라는 세균에 의해 발생합니다. 폐결핵 환자의 80%는 기침과 가래가 생기고, 피로감이나 숨이 차는 증상이 나타나며, 미열도 동반됩니다.

테스트 3
환자 증상: 저는 무릎이 아파요. 무릎을 꿇거나 계단을 오를 때 아픈데요.
cuda:0
AI 응답: 무릎관절에염증이생긴것으로생각됩니다. 염증은무릎관절안에서발생한것이아니라관절주변인대나근육에있는것입니다.

사용자 정의 증상 테스트
----------------------------------------

테스트 1
증상: 머리가 아프고 열이 나요
cuda:0
AI 응답: 머리뼈에 생긴 종양은 뇌하수체종양이 가장 흔합니다. 종양은 크기가 커지면 주변 신경을 압박하여 두통, 시력장애, 청력장애 등을 유발할 수 있습니다.

테스트 2
증상: 기침이 2주째 계속되고 있어요
cuda:0
AI 응답: 흉부X선사진에서 폐렴 소견은 보이지 않으나, 기침과 가래가 지속되는 경우에는 단순한 감기나 기관지염으로 인한 증상인지 아니면 다른 질환에 의한 증상인지 감별진단이 필요합니다. 특히, 50대 이상에서는 폐암을 의심해 볼 수 있으